> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

# 09 — MP両端保持とDE：単独行列のLRA実験
未実行の実験ノートブックです。既存08・08_2と保存結果を変更しません。

**主実験**：Hのみ／L∪H × DEなし／あり。各行列を毎回元モデルから単独変更します。
- E1：L切除と、同数のB・H切除。Bは複数seed。比較個数は `min(|L|, |B|, |H|)`。全L切除も別に保存。
- E2：DE前MP上端のrankを共通予算として、上位保持／下側保護／中央保護。配分5・10・20%。
- E3：DE前後の下側率とPPL改善の対応分析（追加推論不要）。
- 両端保持には、各DE条件内で同rankの上位保持を対照に付けます。

DE前はBEMA、DE後は分散1。λ=s²/n（p≤nへ転置）。既存funcs1.boundsのmatched_upperを使用。
下端TW補正は診断上の選別規則であり、学習済み重みにおける有意性・ノイズ性を保証しません。
正方・near-hard-edgeではLなし。LとHはDE前後で同じ方向を意味しません。

## 実行手順
1. VS CodeでColab GPUカーネルを選択。L4から開始可。全行列・全追加条件は大量のPPL評価になるため、まずMAINを実行。
2. 依存関係を初回のみインストール。APIキーは非表示入力。既存08のDrive出力が必要。
3. 設定→入力確認→軽量テストを実行。`RUN_EXPERIMENTS=True` にして実行セルへ。
4. `PHASES` を追加して再実行すると完了済みの同一maskは再利用。設定の意味が変わると停止し、新しいTAGを要求。
5. 集計セルは実行中断後にも利用可能。PPL・rank・因子化パラメータ数を保存。

大規模実行・LoRAは作成時には実行しません。LoRAは任意セルで、選択した4条件を同一設定で比較します。

In [ ]:
from pathlib import Path
import os, sys, subprocess, getpass
INSTALL_DEPENDENCIES = True
CLONE_IF_MISSING = True
REPO = Path('/content/RMT_utils') if Path('/content').exists() else Path.home() / 'GitHub/RMT_utils'
if not (REPO / 'funcs1.py').exists():
    if not CLONE_IF_MISSING:
        raise FileNotFoundError(f'RMT_utils not found: {REPO}')
    subprocess.run(['git', 'clone', 'https://github.com/ryy1210/RMT_utils', str(REPO)], check=True)
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements-notebooks.txt')], check=True)
sys.path.insert(0, str(REPO))
# 必要なときだけ入力。キーをセル出力・設定ファイルに保存しない。
if not os.environ.get('HF_TOKEN'):
    token = getpass.getpass('HF_TOKEN (cached loginなら空欄): ')
    if token: os.environ['HF_TOKEN'] = token
print('Repository:', REPO)

In [ ]:
import json, hashlib, math, gc, time, re, platform, importlib.metadata, inspect, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import funcs1
from funcs1 import bema_algorithm1_from_eigenvalues, bounds, _dyson_from_svd
MOUNT_DRIVE = True
if MOUNT_DRIVE and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
if Path('/content').exists():
    DATA_ROOT=Path('/content/drive/MyDrive/TUS/hashiguchi/data')
else:
    base=Path.home()/'Library/CloudStorage/GoogleDrive-ryoya.zushi1210@gmail.com'
    candidates=[p/'TUS/hashiguchi/data' for p in base.iterdir() if unicodedata.normalize('NFC',p.name)=='マイドライブ']
    if len(candidates)!=1: raise FileNotFoundError('DATA_ROOTをDriveのdataフォルダに設定してください')
    DATA_ROOT=candidates[0]
LEGACY_OUTPUT=DATA_ROOT/'ibis2026_de_lra'/'ibis2026-de-controlled-v1'
OUTPUT=DATA_ROOT/'ibis2026_de_lra'/'two-edge-lra-v1'
MODELS={'llama':'meta-llama/Llama-3.2-3B','gemma':'google/gemma-3-4b-pt'}
MODULES=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
PHASES=['MAIN']  # 追加実験は ['MAIN','E1','E2']。E3は集計セルで常に作成
ONLY_NAMES=[]  # 空なら全行列。最初の動作確認では1行列名を指定可。結果の互換性に影響しない
RUN_EXPERIMENTS=True
ALL_REQUESTED_COMPLETED=False
BEMA_ALPHA=0.2; TW_BETA=0.1
LOWER_FRACTIONS=[0.05,0.10,0.20]
RANDOM_REPEATS=3; SEED=42
SEQ_LEN=1024; BATCH_SIZE=1
SVD_DEVICE="cuda"  # GPU thin SVD。既存DEの推定式に分解結果を渡して重複SVDを避ける
AUTO_DISCONNECT=False
PUBLISH_WANDB=False
WANDB_ENTITY='ryoya-zushi1210-tokyo-university-of-science'
WANDB_PROJECT='IBIS2026-DE-LRA'

## 保存入力と再開の検証
08の固定checkpointと評価token列を再利用します。旧PPLは再利用せず、同じモデル読み込みでbaselineを再計測。
既存08の所在が違う場合はLEGACY_OUTPUTのみ変更してください。パスを推測して別実験を自動選択しません。
結果は一条件ごとに完了マーカーを最後に書きます。mask・token・実装hashが一致するものだけ再利用します。

In [ ]:
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def digest(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False));tmp.replace(path)
def safe_exp(x): return math.exp(x) if x<709 else float('inf')
def atomic_csv(path,df):
    tmp=path.with_suffix('.tmp.csv');df.to_csv(tmp,index=False);tmp.replace(path)
manifest_path=LEGACY_OUTPUT/'manifest.json'
if not manifest_path.exists(): raise FileNotFoundError(f'08のmanifestがありません: {manifest_path}')
legacy=json.loads(manifest_path.read_text())['settings']
REVISIONS=legacy['revisions'];TOTAL=legacy['total']
assert all(re.fullmatch('[0-9a-f]{40}',REVISIONS[k]) for k in MODELS)
TOKENS={};TOKEN_HASH={}
for key in MODELS:
    assert legacy['models'][key]==MODELS[key]
    path=LEGACY_OUTPUT/f'{key}_tokens.npy'
    receipt=json.loads((LEGACY_OUTPUT/f'{key}_tokens.json').read_text())
    assert receipt['sha256']==sha(path), '08 token checksum mismatch'
    TOKENS[key]=np.load(path)
    assert TOKENS[key].ndim==2 and TOKENS[key].shape[1]==SEQ_LEN
    TOKEN_HASH[key]=sha(path)
OUTPUT.mkdir(parents=True,exist_ok=True)
print('固定token blocks:',{k:len(v) for k,v in TOKENS.items()})

## スペクトル分解と保持mask
フルthin SVDを使用。DE前SVDを既存DE推定式に再利用し、DE後SVDだけ追加します。
保持maskごとに元空間へ逆変換。両端保持は通常TSVDと異なりますが、保持個数以下のrankで因子化可能です。
因子化サイズr(m+n)が元サイズmnを超える場合も隠さず保存し、実効削減率はdenseのままなら0とします。

In [ ]:
def decompose(weight):
    transposed=weight.shape[0]>weight.shape[1]
    w=(weight.T if transposed else weight).float().to(SVD_DEVICE)
    u,s,v=torch.linalg.svd(w,full_matrices=False)
    # 既存DEと同じ推定式。GPU thin SVD結果をfloat64で渡す。
    y,x,z=_dyson_from_svd(w.cpu().double().numpy(),u.cpu().double().numpy(),s.cpu().double().numpy(),v.cpu().double().numpy())
    ud,sd,vd=torch.linalg.svd(torch.as_tensor(y,device=SVD_DEVICE,dtype=torch.float32),full_matrices=False)
    return {False:(u,s,v,None,None,transposed),True:(ud,sd,vd,x,z,transposed)}
def spectral_info(parts,use_de):
    u,s,v,x,z,t=parts;p=len(s);n=v.shape[1]
    eig=s.cpu().double().numpy()**2/n
    fit=bema_algorithm1_from_eigenvalues(eig,p,n,alpha=BEMA_ALPHA,beta=TW_BETA)
    variance=1.0 if use_de else float(fit['sigma2_hat'])
    upper=float(fit['threshold']/fit['sigma2_hat']*variance)
    a,b,sm,sp,c,status,intervals=bounds(p,n,variance,upper)
    _,lower,upper=next(q for q in intervals if q[0]=='matched_upper')
    L=np.flatnonzero(eig<lower);H=np.flatnonzero(eig>upper)
    B=np.flatnonzero((eig>=lower)&(eig<=upper))
    assert len(L)+len(H)+len(B)==p
    return dict(p=p,n=n,sigma2=variance,lower=lower,upper=upper,lower_status=status,L=L,B=B,H=H,
                lower_fraction=len(L)/p,upper_fraction=len(H)/p)
def reconstruct(parts,indices):
    u,s,v,x,z,t=parts
    idx=torch.as_tensor(indices,device=s.device,dtype=torch.long)
    result=(u[:,idx]*s[idx])@v[idx,:]
    if x is not None:
        result*=torch.as_tensor(np.sqrt(x),device=result.device,dtype=result.dtype)[:,None]
        result*=torch.as_tensor(np.sqrt(z),device=result.device,dtype=result.dtype)[None,:]
    if t: result=result.T
    if not torch.isfinite(result).all(): raise FloatingPointError('Nonfinite reconstruction')
    return result

def plans(info,r0,identity):
    p=info['p'];L=info['L'];B=info['B'];H=info['H'];allidx=np.arange(p)
    rows=[]
    def add(phase,label,idx,**extra):
        idx=np.unique(np.asarray(idx,dtype=int))
        assert np.all((idx>=0)&(idx<p))
        rows.append(dict(phase=phase,label=label,indices=idx.tolist(),rank=len(idx),**extra))
    add('MAIN','H',H)
    add('MAIN','LH',np.union1d(L,H))
    add('MAIN','top_same_rank_LH',allidx[:len(L)+len(H)])
    add('E1','remove_all_L',np.setdiff1d(allidx,L),selected=len(L),applicable=bool(len(L)))
    k=min(len(L),len(B),len(H))
    if k:
        add('E1','remove_L_matched',np.setdiff1d(allidx,L[-k:]),selected=k)
        add('E1','remove_H_matched',np.setdiff1d(allidx,H[:k]),selected=k)
        for repeat in range(RANDOM_REPEATS):
            seed=int(digest([SEED,identity,'E1',repeat])[:8],16)
            chosen=np.random.default_rng(seed).choice(B,k,replace=False)
            add('E1',f'remove_B_matched_{repeat}',np.setdiff1d(allidx,chosen),selected=k,random_seed=seed)
    add('E2','top_r0',allidx[:r0],r0=r0)
    for frac in LOWER_FRACTIONS:
        # 上位枠と下側/中央枠の重複を避けるため、候補は元のtop-r0の外側。
        low=L[L>=r0];bulk=B[B>=r0]
        k=min(int(round(frac*r0)),len(low),len(bulk))
        if k==0: continue
        top=allidx[:r0-k]
        add('E2',f'protect_L_{frac}',np.union1d(top,low[-k:]),allocation=frac,actual_k=k,r0=r0)
        for repeat in range(RANDOM_REPEATS):
            seed=int(digest([SEED,identity,'E2',frac,repeat])[:8],16)
            chosen=np.random.default_rng(seed).choice(bulk,k,replace=False)
            add('E2',f'protect_B_{frac}_{repeat}',np.union1d(top,chosen),allocation=frac,actual_k=k,r0=r0,random_seed=seed)
    return rows

## モデル読み込みとPPL評価
08_2の検証付きGemma読み込みを継承。全WikiText-2評価tokenを使用し、PPLのみを表示します。
内部のtoken損失はPPL再集計・対応比較用に保存します。未評価をゼロで埋めません。

In [ ]:
GEMMA_LOADER_VERSION='conditional_verified_v2'
def result_root(key): return OUTPUT/key
def check_loading_info(info):
    if not isinstance(info,dict): raise RuntimeError('Loading report missing')
    errors={k:info.get(k) for k in ['missing_keys','unexpected_keys','mismatched_keys','error_msgs'] if info.get(k)}
    if errors: raise RuntimeError(f'Pretrained checkpoint did not load completely: {errors}')

def extract_gemma_text(full):
    from transformers import Gemma3ForCausalLM
    # 正式クラスでロードした実Tensorを共有。meta上の空モデルへ全モジュールを置換する。
    decoder=full.model.language_model
    with torch.device('meta'):
        text=Gemma3ForCausalLM(full.config.text_config)
    text.model=decoder
    text.lm_head=full.lm_head
    text.generation_config=full.generation_config
    if any(p.is_meta for p in text.parameters()) or any(b.is_meta for b in text.buffers()):
        raise RuntimeError('Unloaded meta tensors remain')
    return text

def model_load(key):
    kwargs=dict(revision=REVISIONS[key],token=os.environ.get('HF_TOKEN'),torch_dtype=torch.bfloat16,
                low_cpu_mem_usage=True,output_loading_info=True)
    if key=='gemma':
        from transformers import Gemma3ForConditionalGeneration
        # 修正: checkpointのarchitectureと一致するクラスを使う。text_configだけを渡さない。
        full,info=Gemma3ForConditionalGeneration.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
        model=extract_gemma_text(full)
        del full; gc.collect()
    else:
        model,info=AutoModelForCausalLM.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
    actual=sum(p.numel() for p in model.parameters())
    if actual!=TOTAL[key]: raise ValueError(f'Parameter count mismatch {actual} != {TOTAL[key]}')
    if any(p.is_meta for p in model.parameters()): raise RuntimeError('Unloaded model parameters')
    root=result_root(key); root.mkdir(parents=True,exist_ok=True)
    write_json(root/'loading_report.json',dict(model=MODELS[key],revision=REVISIONS[key],
        loader_version=GEMMA_LOADER_VERSION if key=='gemma' else 'auto_verified_v2',
        transformers=importlib.metadata.version('transformers'),parameter_count=actual,
        missing_keys=[],unexpected_keys=[],mismatched_keys=[],config_hash=CONFIG_HASH))
    model.eval(); model.config.use_cache=False
    return model.to('cuda')

def evaluate(model,tokens):
    losses=[]; counts=[]
    for start in range(0,len(tokens),BATCH_SIZE):
        ids=torch.as_tensor(tokens[start:start+BATCH_SIZE],device=next(model.parameters()).device)
        logits=model(input_ids=ids,use_cache=False).logits
        sums=torch.zeros(ids.shape[0],device=ids.device,dtype=torch.float64)
        for pos in range(0,ids.shape[1]-1,64):
            end=min(pos+64,ids.shape[1]-1)
            loss=F.cross_entropy(logits[:,pos:end].float().transpose(1,2),ids[:,pos+1:end+1],reduction='none')
            if not torch.isfinite(loss).all(): raise FloatingPointError('Nonfinite NLL')
            sums+=loss.double().sum(1)
        losses.extend(sums.cpu().tolist()); counts.extend([ids.shape[1]-1]*ids.shape[0]); del logits
    return np.asarray(losses),np.asarray(counts)

def resolve(model,name):
    found=[m for k,m in model.named_modules() if isinstance(m,torch.nn.Linear) and (k==name or k.endswith('.'+name))]
    if len(found)!=1: raise ValueError(f'Ambiguous module {name}')
    return found[0]

In [ ]:
evaluate=torch.inference_mode()(evaluate)
def function_fingerprint(f):
    # ipykernelの一時filename・セル番号は再起動で変わるのでhashに含めない。
    import types
    def normalize(code):
        return [code.co_code.hex(),code.co_names,code.co_varnames,
                [normalize(x) if isinstance(x,types.CodeType) else repr(x) for x in code.co_consts]]
    return digest(normalize(f.__code__))
def initialize_manifest():
    settings=dict(version='09-v1',models=MODELS,revisions=REVISIONS,total=TOTAL,token_hash=TOKEN_HASH,
        seq_len=SEQ_LEN,batch=BATCH_SIZE,dtype='bfloat16',svd_device=SVD_DEVICE,bema_alpha=BEMA_ALPHA,
        tw_beta=TW_BETA,fractions=LOWER_FRACTIONS,repeats=RANDOM_REPEATS,seed=SEED,
        funcs1_hash=sha(Path(funcs1.__file__)),modules=MODULES,
        # 関数実装の変更で既存結果を誤用しない。セル内関数のbytecodeを安定的にhash化。
        implementation={('run' if f is _run_v1_for_fingerprint else f.__name__):function_fingerprint(f)
          for f in [decompose,spectral_info,reconstruct,plans,model_load,extract_gemma_text,evaluate.__wrapped__,_run_v1_for_fingerprint,save_result,read_result]})
    global CONFIG_HASH
    CONFIG_HASH=digest(settings)
    path=OUTPUT/'manifest.json'
    if path.exists() and json.loads(path.read_text())['config_hash']!=CONFIG_HASH:
        raise ValueError('設定/実装が保存済み結果と違います。OUTPUTのTAGを変更してください')
    write_json(path,dict(config_hash=CONFIG_HASH,settings=settings))
def read_result(path):
    if not (path/'summary.json').exists(): return None
    obj=json.loads((path/'summary.json').read_text())
    assert obj['config_hash']==CONFIG_HASH
    f=path/'losses.npz'
    assert sha(f)==obj['loss_hash'], 'Result checksum mismatch'
    with np.load(f) as z: loss=z['loss'];count=z['count']
    assert len(loss)==len(TOKENS[obj['model']]) and np.all(count==SEQ_LEN-1) and np.isfinite(loss).all()
    return obj

def save_result(path,key,loss,count,extra):
    path.mkdir(parents=True,exist_ok=True)
    with open(path/'losses.tmp','wb') as f: np.savez_compressed(f,loss=loss,count=count)
    (path/'losses.tmp').replace(path/'losses.npz')
    value=safe_exp(float(loss.sum()/count.sum()))
    obj=dict(model=key,config_hash=CONFIG_HASH,ppl=value if np.isfinite(value) else None,
             ppl_overflow=not np.isfinite(value),loss_hash=sha(path/'losses.npz'),**extra)
    write_json(path/'summary.json',obj)
    return obj

# 再開最適化だけでは実験hashを変えない。旧runはhash互換用で実行しない。
def _run_v1_for_fingerprint():
    global ALL_REQUESTED_COMPLETED
    ALL_REQUESTED_COMPLETED=False
    initialize_manifest()
    if not torch.cuda.is_available(): raise RuntimeError('本番はCUDA GPUカーネルを選択してください')
    if not torch.cuda.is_bf16_supported(): raise RuntimeError('BF16対応GPUが必要です')
    write_json(OUTPUT/'environment.json',dict(gpu=torch.cuda.get_device_name(),python=platform.python_version(),
        versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','datasets']}))
    for key in MODELS:
        model=model_load(key)
        try:
            basepath=OUTPUT/key/'baseline';baseline=read_result(basepath)
            if baseline is None:
                loss,count=evaluate(model,TOKENS[key]);baseline=save_result(basepath,key,loss,count,dict(kind='baseline'))
            if baseline['ppl'] is None or baseline['ppl']>1000: raise RuntimeError('Baseline異常。圧縮前に停止')
            matrices=[(name,module) for name,module in model.named_modules() if isinstance(module,torch.nn.Linear) and name.split('.')[-1] in MODULES]
            if ONLY_NAMES: matrices=[(n,m) for n,m in matrices if n in ONLY_NAMES]
            if not matrices: raise ValueError('対象行列がありません。ONLY_NAMESを確認')
            for j,(name,module) in enumerate(matrices):
                print(f'{key} [{j+1}/{len(matrices)}] {name}',flush=True)
                original=module.weight.detach().cpu().clone()
                parts=decompose(original);infos={d:spectral_info(parts[d],d) for d in [False,True]}
                r0=len(infos[False]['H'])
                for de in [False,True]:
                    info=infos[de];directory=OUTPUT/key/'matrices'/digest(name)[:20]/str(int(de))
                    pp=plans(info,r0,name);selected=[p for p in pp if p['phase'] in PHASES]
                    directory.mkdir(parents=True,exist_ok=True)
                    spectral={k:(v.tolist() if isinstance(v,np.ndarray) else v) for k,v in info.items()}
                    write_json(directory/'plan.json',dict(name=name,de=de,spectrum=spectral,conditions=pp,config_hash=CONFIG_HASH))
                    for condition in selected:
                        idx=condition['indices'];mask_id=digest(idx)[:24]
                        path=directory/mask_id
                        if read_result(path) is not None: continue
                        start=time.time()
                        try:
                            # 全成分保持は無変更にする。SVD丸めだけでbaselineとの差を作らない。
                            approx=original.float().to(module.weight.device) if len(idx)==info['p'] else reconstruct(parts[de],idx)
                            relative=float(torch.linalg.vector_norm(approx.cpu()-original.float())/torch.linalg.vector_norm(original.float()))
                            with torch.no_grad(): module.weight.copy_(approx.to(device=module.weight.device,dtype=module.weight.dtype))
                            del approx
                            loss,count=evaluate(model,TOKENS[key])
                            p,n=info['p'],info['n'];r=len(idx)
                            save_result(path,key,loss,count,dict(name=name,de=de,rank=r,mask_id=mask_id,
                                baseline_ppl=baseline['ppl'],relative_frobenius_error=relative,
                                factor_parameters=r*(p+n),dense_parameters=p*n,
                                effective_reduction=max(0,1-r*(p+n)/(p*n)),model_reduction=max(0,p*n-r*(p+n))/TOTAL[key],
                                seconds=time.time()-start))
                            print(f'  DE={de} {condition["label"]}: rank={r}, saved',flush=True)
                        finally:
                            with torch.no_grad(): module.weight.copy_(original.to(module.weight.device))
                    # 任意LoRAで選択maskを再構成できるようplanを保持。巨大なSVDファイルは保存しない。
                del parts,original;gc.collect();torch.cuda.empty_cache()
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    ALL_REQUESTED_COMPLETED=True

def matrix_results_complete(key,name,shape):
    # 2026-10-21: 保存済みplanと全結果を検証し、完了行列はSVD/DE前に飛ばす。
    # 旧実行のplanも利用可能。片側DEや一部条件だけの完了ではスキップしない。
    saved={}
    for de in [False,True]:
        directory=OUTPUT/key/'matrices'/digest(name)[:20]/str(int(de))
        path=directory/'plan.json'
        if not path.exists(): return False
        obj=json.loads(path.read_text())
        assert obj['config_hash']==CONFIG_HASH, 'Saved plan config mismatch'
        assert obj['name']==name and obj['de']==de, 'Saved plan identity mismatch'
        info=dict(obj['spectrum'])
        assert (info['p'],info['n'])==tuple(sorted(shape)), 'Saved plan shape mismatch'
        for edge in ['L','B','H']: info[edge]=np.asarray(info[edge],dtype=int)
        saved[de]=(directory,obj,info)
    r0=len(saved[False][2]['H'])
    for de,(directory,obj,info) in saved.items():
        # 保存planの欠落を完了と誤認しないよう、軽量な条件生成だけ再実行する。
        expected=plans(info,r0,name)
        assert obj['conditions']==expected, 'Saved plan conditions mismatch'
        selected=[c for c in expected if c['phase'] in PHASES]
        if not selected: return False
        checked=set()
        for condition in selected:
            mask_id=digest(condition['indices'])[:24]
            if mask_id in checked: continue
            result=read_result(directory/mask_id)
            if result is None: return False
            assert (result['model']==key and result['name']==name and result['de']==de
                    and result['mask_id']==mask_id and result['rank']==condition['rank']), 'Saved result identity mismatch'
            checked.add(mask_id)
    return True

def run():
    global ALL_REQUESTED_COMPLETED
    ALL_REQUESTED_COMPLETED=False
    initialize_manifest()
    if not torch.cuda.is_available(): raise RuntimeError('本番はCUDA GPUカーネルを選択してください')
    if not torch.cuda.is_bf16_supported(): raise RuntimeError('BF16対応GPUが必要です')
    write_json(OUTPUT/'environment.json',dict(gpu=torch.cuda.get_device_name(),python=platform.python_version(),
        versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','datasets']}))
    for key in MODELS:
        model=model_load(key)
        try:
            basepath=OUTPUT/key/'baseline';baseline=read_result(basepath)
            if baseline is None:
                loss,count=evaluate(model,TOKENS[key]);baseline=save_result(basepath,key,loss,count,dict(kind='baseline'))
            if baseline['ppl'] is None or baseline['ppl']>1000: raise RuntimeError('Baseline異常。圧縮前に停止')
            matrices=[(name,module) for name,module in model.named_modules() if isinstance(module,torch.nn.Linear) and name.split('.')[-1] in MODULES]
            if ONLY_NAMES: matrices=[(n,m) for n,m in matrices if n in ONLY_NAMES]
            if not matrices: raise ValueError('対象行列がありません。ONLY_NAMESを確認')
            for j,(name,module) in enumerate(matrices):
                print(f'{key} [{j+1}/{len(matrices)}] {name}',flush=True)
                if matrix_results_complete(key,name,tuple(module.weight.shape)):
                    print('  SKIP: 全条件完了（SVD / DE / PPL評価を省略）',flush=True)
                    continue
                original=module.weight.detach().cpu().clone()
                parts=decompose(original);infos={d:spectral_info(parts[d],d) for d in [False,True]}
                r0=len(infos[False]['H'])
                for de in [False,True]:
                    info=infos[de];directory=OUTPUT/key/'matrices'/digest(name)[:20]/str(int(de))
                    pp=plans(info,r0,name);selected=[p for p in pp if p['phase'] in PHASES]
                    directory.mkdir(parents=True,exist_ok=True)
                    spectral={k:(v.tolist() if isinstance(v,np.ndarray) else v) for k,v in info.items()}
                    write_json(directory/'plan.json',dict(name=name,de=de,spectrum=spectral,conditions=pp,config_hash=CONFIG_HASH))
                    for condition in selected:
                        idx=condition['indices'];mask_id=digest(idx)[:24]
                        path=directory/mask_id
                        if read_result(path) is not None: continue
                        start=time.time()
                        try:
                            # 全成分保持は無変更にする。SVD丸めだけでbaselineとの差を作らない。
                            approx=original.float().to(module.weight.device) if len(idx)==info['p'] else reconstruct(parts[de],idx)
                            relative=float(torch.linalg.vector_norm(approx.cpu()-original.float())/torch.linalg.vector_norm(original.float()))
                            with torch.no_grad(): module.weight.copy_(approx.to(device=module.weight.device,dtype=module.weight.dtype))
                            del approx
                            loss,count=evaluate(model,TOKENS[key])
                            p,n=info['p'],info['n'];r=len(idx)
                            save_result(path,key,loss,count,dict(name=name,de=de,rank=r,mask_id=mask_id,
                                baseline_ppl=baseline['ppl'],relative_frobenius_error=relative,
                                factor_parameters=r*(p+n),dense_parameters=p*n,
                                effective_reduction=max(0,1-r*(p+n)/(p*n)),model_reduction=max(0,p*n-r*(p+n))/TOTAL[key],
                                seconds=time.time()-start))
                            print(f'  DE={de} {condition["label"]}: rank={r}, saved',flush=True)
                        finally:
                            with torch.no_grad(): module.weight.copy_(original.to(module.weight.device))
                    # 任意LoRAで選択maskを再構成できるようplanを保持。巨大なSVDファイルは保存しない。
                del parts,original;gc.collect();torch.cuda.empty_cache()
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    ALL_REQUESTED_COMPLETED=True


## 軽量検証（モデル取得・GPU推論なし）
長方・転置行列の全成分復元、rank・mask・DE逆変換を検証します。

In [ ]:
def smoke_test():
    global SVD_DEVICE
    previous=SVD_DEVICE;SVD_DEVICE='cpu'
    try:
        torch.manual_seed(17)
        for shape in [(24,80),(80,24),(24,24)]:
            w=torch.randn(*shape)
            parts=decompose(w)
            for de in [False,True]:
                rebuilt=reconstruct(parts[de],np.arange(min(shape)))
                assert torch.allclose(w,rebuilt,atol=3e-5,rtol=3e-5)
                assert torch.count_nonzero(reconstruct(parts[de],[]))==0
                inf=spectral_info(parts[de],de)
                assert len(inf['L'])+len(inf['B'])+len(inf['H'])==min(shape)
        fake=dict(p=20,L=np.arange(16,20),B=np.arange(4,16),H=np.arange(4))
        jobs=plans(fake,4,'synthetic')
        for job in jobs:
            assert job['rank']==len(set(job['indices']))
            if job['phase']=='E2': assert job['rank']==4
        assert plans(fake,4,'synthetic')==jobs
        print('PASS: DE逆変換・転置・ゼロrank・mask・共通rank・再現性')
    finally: SVD_DEVICE=previous
smoke_test()

## 本番（ここからモデルの取得・GPU計算）
MAINだけで1行列あたり最大6条件＋baseline。Lなしでは同じmaskの条件は再推論しません。
E1/E2は条件を増やすため、MAIN完了後にPHASESへ追加する運用を推奨。
途中再開は保存済みPPLをスキップします。SVDは行列ごとに再計算しますが、同一実行中は共有します。

In [ ]:
if RUN_EXPERIMENTS:
    run()
else:
    print('本番未実行。設定のRUN_EXPERIMENTS=Trueで開始します。')

## 結果・E3の対応分析
phase別の予定件数と完了件数を表示。DEの有無・保持方法を同じ行列で対応付けます。
正のgainはPPL改善。rankが違う比較と同rank比較を区別します。E1/E2も全結果CSVへ保存。

In [ ]:
initialize_manifest()
rows=[]
for path in OUTPUT.glob('*/matrices/*/*/plan.json'):
    plan=json.loads(path.read_text());assert plan['config_hash']==CONFIG_HASH
    key=path.relative_to(OUTPUT).parts[0]
    for c in plan['conditions']:
        result=read_result(path.parent/digest(c['indices'])[:24])
        row=dict(model=key,name=plan['name'],de=plan['de'],phase=c['phase'],label=c['label'],rank=c['rank'],
            lower_fraction=plan['spectrum']['lower_fraction'],lower_status=plan['spectrum']['lower_status'],
            status='complete' if result else 'pending',**{k:v for k,v in c.items() if k in ['allocation','actual_k','selected','applicable']})
        if result: row.update({k:v for k,v in result.items() if k not in ['config_hash','model','name','de','rank']})
        rows.append(row)
RESULTS=pd.DataFrame(rows)
if not RESULTS.empty:
    RESULTS['module']=RESULTS.name.str.split('.').str[-1]
    atomic_csv(OUTPUT/'results.csv',RESULTS)
    display(RESULTS.groupby(['model','phase','status']).size().rename('conditions').reset_index())
    done=RESULTS[RESULTS.status=='complete'].copy()
    if 'ppl_overflow' in done and done.ppl_overflow.fillna(False).any():
        print('注意: PPLが数値範囲を超えた条件があります。欠損表示を良好な結果と解釈しないでください。')
    main=done[done.phase=='MAIN']
    wide=main.pivot(index=['model','name','de'],columns='label',values='ppl')
    if {'LH','H','top_same_rank_LH'}<=set(wide):
        wide['lower_addition_gain']=wide.H-wide.LH
        wide['same_rank_gain']=wide.top_same_rank_LH-wide.LH
        wide=wide.join(main.drop_duplicates(['model','name','de']).set_index(['model','name','de'])[['lower_fraction']])
        atomic_csv(OUTPUT/'E3_within_DE.csv',wide.reset_index())
        display(wide.groupby(['model','de'])[['lower_addition_gain','same_rank_gain']].median())
        paired=wide.reset_index().pivot(index=['model','name'],columns='de',values=['LH','H','lower_fraction','same_rank_gain'])
        paired.columns=[f'{a}_{"DE" if b else "raw"}' for a,b in paired.columns]
        if {'LH_raw','LH_DE','lower_fraction_raw','lower_fraction_DE'}<=set(paired):
            paired['DE_PPL_gain_LH']=paired.LH_raw-paired.LH_DE
            paired['lower_fraction_drop']=paired.lower_fraction_raw-paired.lower_fraction_DE
            atomic_csv(OUTPUT/'E3_between_DE.csv',paired.reset_index())
            ranks=main.pivot(index=['model','name'],columns=['label','de'],values='rank')
            ranks.columns=[f'rank_{a}_{"DE" if b else "raw"}' for a,b in ranks.columns]
            paired=paired.join(ranks)
            atomic_csv(OUTPUT/'E3_between_DE.csv',paired.reset_index())
            display(paired.head(10))
    for key,g in main.groupby('model'):
        fig,ax=plt.subplots(figsize=(8,5))
        for (de,label),h in g.groupby(['de','label']):
            ax.scatter(h.effective_reduction*100,h.ppl,s=16,alpha=.6,label=f'{label}, DE={de}')
        ax.set(xlabel='Effective factorized reduction in the matrix (%)',ylabel='PPL',yscale='log',title=key)
        ax.legend(fontsize=8);fig.tight_layout();fig.savefig(OUTPUT/f'{key}_main_ppl.png',dpi=160);plt.show()
else: print('まだ実験結果はありません。')
# E1：同数切除のPPL差。E2：共通rankのtop保持に対するPPL改善。
if not RESULTS.empty:
    done=RESULTS[RESULTS.status=='complete'].copy()
    if 'ppl_overflow' in done and done.ppl_overflow.fillna(False).any():
        print('注意: PPLが数値範囲を超えた条件があります。欠損表示を良好な結果と解釈しないでください。')
    for phase,reference in [('E1','remove_L_matched'),('E2','top_r0')]:
        subset=done[done.phase==phase]
        if subset.empty: continue
        ref=subset[subset.label==reference][['model','name','de','ppl']].rename(columns={'ppl':'reference_ppl'})
        comparisons=subset.merge(ref,on=['model','name','de'],how='inner',validate='many_to_one')
        comparisons['PPL_gain_vs_reference']=comparisons.reference_ppl-comparisons.ppl
        atomic_csv(OUTPUT/f'{phase}_paired.csv',comparisons)
        display(comparisons.groupby(['model','module','de','label'])['PPL_gain_vs_reference'].agg(['count','median']))
        # E1: 正ならL切除より被害が少ない。E2: 正ならtop-r0より良い。


## 任意：LoRA前後の比較
MAIN完了後、検証データで選んだ行列名を `LORA_NAMES` に指定します。評価test PPLで最良条件を選別しないでください。
同一行列のH/LH × DE有無を同じ学習設定で比較。WikiText-2 trainで1 epoch、評価は固定test。
既定OFF。学習は1条件ごとにモデルを再ロード。中断した学習はその条件を最初からやり直します。

In [ ]:
RUN_LORA=False
LORA_NAMES={'llama':[],'gemma':[]}
LORA_SETTINGS=dict(r=8,lora_alpha=16,lora_dropout=0.05,learning_rate=2e-4,epochs=1,seed=42)
if RUN_LORA:
    from peft import LoraConfig,get_peft_model
    from transformers import Trainer,TrainingArguments,default_data_collator,set_seed
    from datasets import Dataset
    initialize_manifest()
    for key,names in LORA_NAMES.items():
        if not names: continue
        tokenizer=AutoTokenizer.from_pretrained(MODELS[key],revision=REVISIONS[key],token=os.environ.get('HF_TOKEN'))
        train=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split='train')
        ids=tokenizer('\n\n'.join(train['text']),add_special_tokens=False)['input_ids']
        blocks=np.asarray(ids[:len(ids)//SEQ_LEN*SEQ_LEN]).reshape(-1,SEQ_LEN)
        dataset=Dataset.from_dict(dict(input_ids=blocks.tolist(),labels=blocks.tolist()))
        for name in names:
            for de in [False,True]:
                plan_path=OUTPUT/key/'matrices'/digest(name)[:20]/str(int(de))/'plan.json'
                plan=json.loads(plan_path.read_text())
                assert plan['config_hash']==CONFIG_HASH
                for label in ['H','LH']:
                    c=next(c for c in plan['conditions'] if c['phase']=='MAIN' and c['label']==label)
                    before=read_result(plan_path.parent/digest(c['indices'])[:24])
                    if before is None: raise RuntimeError('MAINの評価を先に完了してください')
                    signature=digest([CONFIG_HASH,name,de,label,LORA_SETTINGS,train._fingerprint])[:24]
                    dest=OUTPUT/key/'lora'/signature
                    if read_result(dest): continue
                    set_seed(LORA_SETTINGS['seed']);model=model_load(key)
                    try:
                        module=resolve(model,name);parts=decompose(module.weight.detach().cpu())
                        value=reconstruct(parts[de],c['indices'])
                        with torch.no_grad():module.weight.copy_(value.to(module.weight.device,dtype=module.weight.dtype))
                        del parts,value
                        model=get_peft_model(model,LoraConfig(r=LORA_SETTINGS['r'],lora_alpha=LORA_SETTINGS['lora_alpha'],
                            lora_dropout=LORA_SETTINGS['lora_dropout'],target_modules=MODULES,task_type='CAUSAL_LM'))
                        args=TrainingArguments(output_dir=str(dest/'trainer'),num_train_epochs=LORA_SETTINGS['epochs'],
                            per_device_train_batch_size=1,gradient_accumulation_steps=2,learning_rate=LORA_SETTINGS['learning_rate'],
                            bf16=True,save_strategy='no',report_to=[],logging_steps=20,seed=LORA_SETTINGS['seed'],
                            gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},optim='adamw_torch')
                        trainer=Trainer(model=model,args=args,train_dataset=dataset,data_collator=default_data_collator)
                        trainer.train();model.eval()
                        loss,count=evaluate(model,TOKENS[key])
                        model.save_pretrained(dest/'adapter')
                        save_result(dest,key,loss,count,dict(name=name,de=de,label=label,before_ppl=before['ppl'],
                            train_fingerprint=train._fingerprint,lora_settings=LORA_SETTINGS))
                        del trainer
                    finally:
                        del model;gc.collect();torch.cuda.empty_cache()
lora_rows=[]
for path in OUTPUT.glob('*/lora/*/summary.json'):
    obj=read_result(path.parent)
    if obj: lora_rows.append(obj)
if lora_rows:
    LORA_RESULTS=pd.DataFrame(lora_rows)
    atomic_csv(OUTPUT/'lora_results.csv',LORA_RESULTS)
    display(LORA_RESULTS[['model','name','de','label','before_ppl','ppl']])


In [ ]:
if PUBLISH_WANDB and not RESULTS.empty:
    import wandb
    if not os.environ.get('WANDB_API_KEY'):
        os.environ['WANDB_API_KEY']=getpass.getpass('WANDB_API_KEY: ')
    with wandb.init(entity=WANDB_ENTITY,project=WANDB_PROJECT,job_type='two-edge-lra',config={'config_hash':CONFIG_HASH}) as wb:
        wb.log({'results':wandb.Table(dataframe=RESULTS.fillna(''))})
if AUTO_DISCONNECT and RUN_EXPERIMENTS and ALL_REQUESTED_COMPLETED:
    # 計算セルが例外で止まった場合はRun Allもここに到達しません。
    from google.colab import runtime
    runtime.unassign()